In [ ]:
import math
from dataclasses import dataclass
from typing import List, Dict, Tuple, Optional

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from torch.optim import AdamW


# =========================
# 1. 설정
# =========================
@dataclass
class Config:
    pretrained_model_name: str = "bert-base-multilingual-cased"
    max_length: int = 128
    batch_size: int = 8
    lr: float = 2e-5
    epochs: int = 3
    inner_dim: int = 64
    device: str = "cuda" if torch.cuda.is_available() else "cpu"


# =========================
# 2. 라벨 매핑
# =========================
label2id = {
    "PER": 0,
    "ORG": 1,
    "LOC": 2,
}
id2label = {v: k for k, v in label2id.items()}


# =========================
# 3. 예시 데이터
#    entity의 start/end는 "문자(char) 단위 인덱스"
# =========================
train_data = [
    {
        "text": "나는 삼성전자에서 일한다.",
        "entities": [
            {"start": 3, "end": 7, "label": "ORG"}  # "삼성전자"
        ]
    },
    {
        "text": "이순신은 조선의 장군이다.",
        "entities": [
            {"start": 0, "end": 2, "label": "PER"},  # "이순신"
            {"start": 5, "end": 6, "label": "ORG"}   # "조선" (예시용)
        ]
    },
    {
        "text": "서울은 한국의 수도다.",
        "entities": [
            {"start": 0, "end": 1, "label": "LOC"},  # "서울"
            {"start": 4, "end": 5, "label": "LOC"}   # "한국"
        ]
    },
]


# =========================
# 4. Loss 함수
#    GlobalPointer 구현에서 자주 쓰는 multilabel categorical crossentropy
# =========================
def multilabel_categorical_crossentropy(y_pred: torch.Tensor, y_true: torch.Tensor) -> torch.Tensor:
    """
    y_pred: (batch, ent_type_size, seq_len, seq_len)
    y_true: same shape, values in {0,1}
    """
    y_pred = (1 - 2 * y_true) * y_pred
    y_pred_neg = y_pred - y_true * 1e12
    y_pred_pos = y_pred - (1 - y_true) * 1e12

    zeros = torch.zeros_like(y_pred[..., :1])
    y_pred_neg = torch.cat([y_pred_neg, zeros], dim=-1)
    y_pred_pos = torch.cat([y_pred_pos, zeros], dim=-1)

    neg_loss = torch.logsumexp(y_pred_neg, dim=-1)
    pos_loss = torch.logsumexp(y_pred_pos, dim=-1)

    return (neg_loss + pos_loss).mean()


# =========================
# 5. RoPE (Rotary Position Embedding)
#    GlobalPointer 논문 계열 구현에서 자주 포함되는 위치 인코딩
# =========================
def sinusoidal_position_embedding(batch_size: int, seq_len: int, output_dim: int, device: str):
    position_ids = torch.arange(0, seq_len, dtype=torch.float).unsqueeze(-1)  # (seq_len, 1)
    indices = torch.arange(0, output_dim // 2, dtype=torch.float)
    indices = torch.pow(10000, -2 * indices / output_dim)  # (output_dim//2,)

    embeddings = position_ids * indices  # (seq_len, output_dim//2)
    sin_embeddings = torch.sin(embeddings)
    cos_embeddings = torch.cos(embeddings)

    # interleave sin/cos
    pos_emb = torch.stack([sin_embeddings, cos_embeddings], dim=-1)  # (seq_len, output_dim//2, 2)
    pos_emb = pos_emb.reshape(seq_len, output_dim)  # (seq_len, output_dim)
    pos_emb = pos_emb.unsqueeze(0).repeat(batch_size, 1, 1)  # (batch_size, seq_len, output_dim)
    return pos_emb.to(device)


def apply_rope(x: torch.Tensor, pos_emb: torch.Tensor) -> torch.Tensor:
    """
    x: (batch, seq_len, ent_type_size, inner_dim)
    pos_emb: (batch, seq_len, inner_dim)
    """
    cos_pos = pos_emb[..., 1::2].repeat_interleave(2, dim=-1)  # (batch, seq_len, inner_dim)
    sin_pos = pos_emb[..., ::2].repeat_interleave(2, dim=-1)

    x2 = torch.stack([-x[..., 1::2], x[..., ::2]], dim=-1).reshape_as(x)
    return x * cos_pos.unsqueeze(2) + x2 * sin_pos.unsqueeze(2)


# =========================
# 6. GlobalPointer Head
# =========================
class GlobalPointer(nn.Module):
    def __init__(self, hidden_size: int, ent_type_size: int, inner_dim: int, use_rope: bool = True):
        super().__init__()
        self.ent_type_size = ent_type_size
        self.inner_dim = inner_dim
        self.use_rope = use_rope

        # 각 토큰 hidden state -> [ent_type_size * inner_dim * 2]
        # 이후 qw, kw 로 반씩 나눔
        self.dense = nn.Linear(hidden_size, ent_type_size * inner_dim * 2)

    def forward(self, last_hidden_state: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        """
        last_hidden_state: (batch, seq_len, hidden_size)
        attention_mask:    (batch, seq_len)
        return logits:     (batch, ent_type_size, seq_len, seq_len)
        """
        batch_size, seq_len, _ = last_hidden_state.size()

        # (batch, seq_len, ent_type_size * inner_dim * 2)
        outputs = self.dense(last_hidden_state)

        # (batch, seq_len, ent_type_size, inner_dim * 2)
        outputs = outputs.view(batch_size, seq_len, self.ent_type_size, self.inner_dim * 2)

        # (batch, seq_len, ent_type_size, inner_dim)
        qw, kw = outputs[..., :self.inner_dim], outputs[..., self.inner_dim:]

        if self.use_rope:
            pos_emb = sinusoidal_position_embedding(
                batch_size=batch_size,
                seq_len=seq_len,
                output_dim=self.inner_dim,
                device=last_hidden_state.device
            )
            qw = apply_rope(qw, pos_emb)
            kw = apply_rope(kw, pos_emb)

        # 핵심:
        # score[c, i, j] = <qw[c, i], kw[c, j]>
        # einsum 결과 shape: (batch, ent_type_size, seq_len, seq_len)
        logits = torch.einsum("bmhd,bnhd->bhmn", qw, kw)

        # scale
        logits = logits / math.sqrt(self.inner_dim)

        # padding mask
        # attention_mask: (batch, seq_len)
        pad_mask = attention_mask.unsqueeze(1).unsqueeze(1) * attention_mask.unsqueeze(1).unsqueeze(2)
        # (batch, 1, seq_len, seq_len)
        logits = logits * pad_mask - (1 - pad_mask) * 1e12

        # end < start 인 span 제거 (하삼각 마스킹)
        mask = torch.tril(torch.ones(seq_len, seq_len, device=logits.device), diagonal=-1)
        logits = logits - mask.unsqueeze(0).unsqueeze(0) * 1e12

        return logits


# =========================
# 7. BERT + GlobalPointer
# =========================
class BertGlobalPointerForNER(nn.Module):
    def __init__(self, pretrained_model_name: str, ent_type_size: int, inner_dim: int):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(pretrained_model_name)
        hidden_size = self.encoder.config.hidden_size
        self.global_pointer = GlobalPointer(
            hidden_size=hidden_size,
            ent_type_size=ent_type_size,
            inner_dim=inner_dim,
            use_rope=True,
        )

    def forward(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
        labels: Optional[torch.Tensor] = None
    ) -> Dict[str, torch.Tensor]:
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden_state = outputs.last_hidden_state  # (batch, seq_len, hidden_size)

        logits = self.global_pointer(last_hidden_state, attention_mask)

        result = {"logits": logits}
        if labels is not None:
            loss = multilabel_categorical_crossentropy(logits, labels)
            result["loss"] = loss
        return result


# =========================
# 8. Dataset
#    char span -> token span 변환 후 label matrix 생성
# =========================
class NERDataset(Dataset):
    def __init__(self, data: List[Dict], tokenizer, label2id: Dict[str, int], max_length: int):
        self.data = data
        self.tokenizer = tokenizer
        self.label2id = label2id
        self.max_length = max_length
        self.ent_type_size = len(label2id)

    def __len__(self):
        return len(self.data)

    def _char_to_token_span(self, offset_mapping, char_start: int, char_end: int) -> Tuple[Optional[int], Optional[int]]:
        """
        char_start, char_end 는 inclusive char index라고 가정
        offset_mapping: [(start_char, end_char), ...]
        반환값은 token_start, token_end
        """
        token_start, token_end = None, None

        for idx, (s, e) in enumerate(offset_mapping):
            if s == e:  # special token or padding
                continue

            # 시작 문자가 이 토큰 안에 포함되는지
            if s <= char_start < e and token_start is None:
                token_start = idx

            # 끝 문자가 이 토큰 안에 포함되는지
            if s <= char_end < e:
                token_end = idx
                break

        return token_start, token_end

    def __getitem__(self, idx):
        item = self.data[idx]
        text = item["text"]
        entities = item["entities"]

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_offsets_mapping=True,
            return_tensors="pt"
        )

        input_ids = encoding["input_ids"].squeeze(0)          # (seq_len,)
        attention_mask = encoding["attention_mask"].squeeze(0)  # (seq_len,)
        offset_mapping = encoding["offset_mapping"].squeeze(0).tolist()

        seq_len = input_ids.size(0)
        labels = torch.zeros((self.ent_type_size, seq_len, seq_len), dtype=torch.float)

        for ent in entities:
            char_start = ent["start"]
            char_end = ent["end"]  # inclusive
            label_id = self.label2id[ent["label"]]

            token_start, token_end = self._char_to_token_span(offset_mapping, char_start, char_end)
            if token_start is None or token_end is None:
                continue
            if token_start >= seq_len or token_end >= seq_len:
                continue
            if token_end < token_start:
                continue

            labels[label_id, token_start, token_end] = 1.0

        return {
            "text": text,
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
            "offset_mapping": offset_mapping,
        }


# =========================
# 9. collate_fn
# =========================
def collate_fn(batch):
    return {
        "texts": [x["text"] for x in batch],
        "input_ids": torch.stack([x["input_ids"] for x in batch], dim=0),
        "attention_mask": torch.stack([x["attention_mask"] for x in batch], dim=0),
        "labels": torch.stack([x["labels"] for x in batch], dim=0),
        "offset_mappings": [x["offset_mapping"] for x in batch],
    }


# =========================
# 10. 디코딩
# =========================
def decode_entities(
    text: str,
    logits: torch.Tensor,
    offset_mapping: List[Tuple[int, int]],
    id2label: Dict[int, str],
    threshold: float = 0.0
):
    """
    logits: (ent_type_size, seq_len, seq_len)
    """
    entities = []
    ent_type_size, seq_len, _ = logits.shape

    for ent_type_id in range(ent_type_size):
        label_name = id2label[ent_type_id]
        for start_idx in range(seq_len):
            for end_idx in range(start_idx, seq_len):
                score = logits[ent_type_id, start_idx, end_idx].item()
                if score <= threshold:
                    continue

                start_char, _ = offset_mapping[start_idx]
                _, end_char = offset_mapping[end_idx]

                # special token / padding 제외
                if start_char == end_char:
                    continue

                entity_text = text[start_char:end_char]
                entities.append({
                    "label": label_name,
                    "start_token": start_idx,
                    "end_token": end_idx,
                    "start_char": start_char,
                    "end_char_exclusive": end_char,
                    "text": entity_text,
                    "score": round(score, 4),
                })

    # 점수순 정렬
    entities = sorted(entities, key=lambda x: x["score"], reverse=True)
    return entities


# =========================
# 11. 평가용 간단 F1
# =========================
def extract_gold_entities(labels: torch.Tensor, id2label: Dict[int, str]):
    """
    labels: (ent_type_size, seq_len, seq_len)
    """
    results = set()
    ent_type_size, seq_len, _ = labels.shape
    for c in range(ent_type_size):
        for i in range(seq_len):
            for j in range(i, seq_len):
                if labels[c, i, j].item() > 0:
                    results.add((id2label[c], i, j))
    return results


def extract_pred_entities(logits: torch.Tensor, id2label: Dict[int, str], threshold: float = 0.0):
    """
    logits: (ent_type_size, seq_len, seq_len)
    """
    results = set()
    ent_type_size, seq_len, _ = logits.shape
    for c in range(ent_type_size):
        for i in range(seq_len):
            for j in range(i, seq_len):
                if logits[c, i, j].item() > threshold:
                    results.add((id2label[c], i, j))
    return results


def compute_f1(model, dataloader, device, id2label, threshold=0.0):
    model.eval()
    tp, fp, fn = 0, 0, 0

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs["logits"]

            for b in range(input_ids.size(0)):
                gold = extract_gold_entities(labels[b].cpu(), id2label)
                pred = extract_pred_entities(logits[b].cpu(), id2label, threshold=threshold)

                tp += len(gold & pred)
                fp += len(pred - gold)
                fn += len(gold - pred)

    precision = tp / (tp + fp + 1e-12)
    recall = tp / (tp + fn + 1e-12)
    f1 = 2 * precision * recall / (precision + recall + 1e-12)
    return precision, recall, f1


# =========================
# 12. 학습
# =========================
def train():
    cfg = Config()

    tokenizer = AutoTokenizer.from_pretrained(cfg.pretrained_model_name)
    dataset = NERDataset(train_data, tokenizer, label2id, cfg.max_length)
    dataloader = DataLoader(dataset, batch_size=cfg.batch_size, shuffle=True, collate_fn=collate_fn)

    model = BertGlobalPointerForNER(
        pretrained_model_name=cfg.pretrained_model_name,
        ent_type_size=len(label2id),
        inner_dim=cfg.inner_dim
    ).to(cfg.device)

    optimizer = AdamW(model.parameters(), lr=cfg.lr)

    for epoch in range(cfg.epochs):
        model.train()
        total_loss = 0.0

        for batch in dataloader:
            input_ids = batch["input_ids"].to(cfg.device)
            attention_mask = batch["attention_mask"].to(cfg.device)
            labels = batch["labels"].to(cfg.device)

            optimizer.zero_grad()

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )
            loss = outputs["loss"]
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(dataloader)
        precision, recall, f1 = compute_f1(model, dataloader, cfg.device, id2label, threshold=0.0)

        print(f"[Epoch {epoch+1}/{cfg.epochs}] loss={avg_loss:.4f} "
              f"precision={precision:.4f} recall={recall:.4f} f1={f1:.4f}")

    return model, tokenizer, cfg


# =========================
# 13. 추론
# =========================
def predict(model, tokenizer, cfg, text: str, threshold: float = 0.0):
    model.eval()

    encoding = tokenizer(
        text,
        truncation=True,
        padding="max_length",
        max_length=cfg.max_length,
        return_offsets_mapping=True,
        return_tensors="pt"
    )

    input_ids = encoding["input_ids"].to(cfg.device)
    attention_mask = encoding["attention_mask"].to(cfg.device)
    offset_mapping = encoding["offset_mapping"].squeeze(0).tolist()

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs["logits"].squeeze(0).cpu()  # (ent_type_size, seq_len, seq_len)

    entities = decode_entities(
        text=text,
        logits=logits,
        offset_mapping=offset_mapping,
        id2label=id2label,
        threshold=threshold
    )
    return entities


# =========================
# 14. 실행 예시
# =========================
if __name__ == "__main__":
    model, tokenizer, cfg = train()

    test_text = "나는 서울에서 삼성전자 연구원을 만났다."
    preds = predict(model, tokenizer, cfg, test_text, threshold=0.0)

    print("\n[Prediction]")
    for ent in preds[:20]:
        print(ent)

# ipynb code

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[Epoch 1/3] loss=0.1524 precision=0.0000 recall=0.0000 f1=0.0000
[Epoch 2/3] loss=0.1449 precision=0.0000 recall=0.0000 f1=0.0000
[Epoch 3/3] loss=0.1341 precision=0.0000 recall=0.0000 f1=0.0000

[Prediction]
